# Forgetful bot

In [ ]:
import os
from openai import OpenAI 
from dotenv import load_dotenv

load_dotenv()

# Ensure your .env has both OPENAI_API_KEY and BASE_URL
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"), base_url=os.getenv("BASE_URL"))

print("Chatbot ready (with no memory)! Type 'quit' or 'exit' to end.\n")

# --- STARTUP SEQUENCE ---
# The bot asks these questions, but it won't save your answers.
print("Bot: Hello! Who are you?")
user_name = input("You: ")

print(f"Bot: Nice to meet you, {user_name}! Tell me some of your features or hobbies.")
user_features = input("You: ")

print("Bot: Thanks for sharing! Now you can ask me anything.\n")
# -------------------------

while True:
    user_input = input("You: ")

    if user_input.lower() in ["quit", "exit"]:
        print("Goodbye!")
        break

    # We send ONLY the current input. 
    # The 'user_name' and 'user_features' variables above are NOT sent to the API.
    completion = client.chat.completions.create(
        model= os.getenv("MODEL_NAME"),
        messages=[{"role": "user", "content": user_input}],
    )

    bot_response = completion.choices[0].message.content
    print(f"Bot: {bot_response}\n")

Chatbot ready (with no memory)! Type 'quit' or 'exit' to end.

Bot: Hello! Who are you?
Bot: Nice to meet you, fateme! Tell me some of your features or hobbies.
Bot: Thanks for sharing! Now you can ask me anything.

Goodbye!


# A long term memory  bot

In [ ]:
import os
import time
from openai import OpenAI 
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"), base_url=os.getenv("BASE_URL"))
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "long-term-memory"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536, 
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

def get_embedding(text):
    res = client.embeddings.create(input=text, model="text-embedding-3-small")
    return res.data[0].embedding

def save_to_memory(user_text, bot_text):
    """Helper function to save interactions to Pinecone."""
    timestamp = str(time.time())
    full_text = f"User said: {user_text} | Bot replied: {bot_text}"
    index.upsert(vectors=[(timestamp, get_embedding(full_text), {"text": full_text})])

# # --- ONCE: INTRODUCTION SEQUENCE ---
print("Chatbot ready! (Starting setup...)\n")

name = input("Bot: Hello! Who are you?\nYou: ")
features = input(f"Bot: Nice to meet you, {name}! Tell me some of your features or hobbies.\nYou: ")

# Immediately save these to Pinecone so the bot "learns" them
save_to_memory("My name is " + name, "I will remember that your name is " + name)
save_to_memory("My features/hobbies are: " + features, "Got it, I've noted your features.")

print("\nBot: Thanks for sharing! Now you can ask me anything. I will remember our past.")
# -----------------------------------

# --- LOOP: MAIN CONVERSATION ---
while True:
    user_input = input("\nYou: ")
    
    if user_input.lower() in ["quit", "exit"]:
        print("Bot: Goodbye! I'll remember our chat next time.")
        break

    # 1. RECALL: Look for relevant memories (like the name or features we just saved)
    query_vector = get_embedding(user_input)
    search_results = index.query(vector=query_vector, top_k=3, include_metadata=True)
    
    context = ""
    for match in search_results["matches"]:
        context += f"\nPast interaction: {match['metadata']['text']}"

    # 2. GENERATE: Respond using the context
    completion = client.chat.completions.create(
        model=os.getenv("MODEL_NAME"),
        messages=[
            {"role": "system", "content": f"You are a helpful assistant with long-term memory. Use this context to recognize the user: {context}"},
            {"role": "user", "content": user_input}
        ],
    )

    bot_response = completion.choices[0].message.content
    print(f"Bot: {bot_response}")

    # 3. SAVE: Store the current exchange
    save_to_memory(user_input, bot_response)